# Gemma 4 Developer Agent Simulator

## 1. Environment Configuration and Package Installation

In [1]:
import glob
import importlib
import os
import shutil
import subprocess
import sys
from pathlib import Path

# Configure environment variables for offline vLLM serving and LiteLLM routing
os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS'] = '1'
os.environ['VLLM_ENGINE_READY_TIMEOUT_S'] = '1200'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['OTEL_SDK_DISABLED'] = 'true'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')


assert WHEELHOUSE_DIR.exists(), (
    f'Wheelhouse not found at {WHEELHOUSE_DIR}. '
    'Attach the wheelhouse dataset in the Input panel (check !ls /kaggle/input/datasets).'
)

# Remove broken cutlass .pth hooks if present
for pth_pattern in (
    '/usr/local/lib/python*/dist-packages/*cutlass*.pth',
    '/usr/local/lib/python*/site-packages/*cutlass*.pth',
):
    for pth in glob.glob(pth_pattern):
        try:
            os.unlink(pth)
        except OSError:
            pass

# Restore PEP 440 '+cu128' wheel filenames stripped by Kaggle dataset uploads
tmp_whl = Path('/tmp/wheelhouse')
tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower():
        continue
    target_name = (
        w.name.replace('cu128', '+cu128')
        if ('cu128' in w.name and '+' not in w.name)
        else w.name
    )
    target = tmp_whl / target_name
    if not target.exists():
        os.symlink(w, target)

wheels = sorted(str(w) for w in tmp_whl.glob('*.whl'))
print(f'Installing {len(wheels)} wheels from {WHEELHOUSE_DIR}...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels],
    check=True,
)
importlib.invalidate_caches()
print('Wheelhouse installation complete.')

Installing 41 wheels from /kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse...
Wheelhouse installation complete.


## 2. Competition Dataset and Sample Submission

In [2]:
# --- Guard: the install cell must have run first ---
try:
    import swegemma
except ModuleNotFoundError:
    raise RuntimeError(
        "swegemma is not installed. Run the 'Environment Configuration and Package "
        "Installation' cell first, wait for 'Wheelhouse installation complete.', "
        "then rerun this cell."
    )

from swegemma.models import load_tasks
import zipfile
import shutil
from pathlib import Path

DATA_DIR = Path('/kaggle/input/competitions/gemma-4-developer-agent')
WORKING_DIR = Path('/kaggle/working')
WORKING_DIR.mkdir(parents=True, exist_ok=True)

# --- Locate YOUR submission dataset (the attached dataset that contains agent.yaml) ---
# If auto-detection picks the wrong folder, set it by hand, e.g.:
# MY_SUBMISSION_SRC = Path('/kaggle/input/datasets/<owner>/<slug>')
MY_SUBMISSION_SRC = None

if MY_SUBMISSION_SRC is None:
    INPUT_ROOT = Path('/kaggle/input')
    patterns = [
        'datasets/*/*/agent.yaml',
        'datasets/*/*/*/agent.yaml',   # in case it sits inside a wrapper folder
        '*/agent.yaml',
        '*/*/agent.yaml',
    ]
    hits = []
    for pat in patterns:
        hits += [
            p for p in INPUT_ROOT.glob(pat)
            if 'competitions' not in p.parts and 'sample_submission' not in p.parts
        ]

    if hits:
        MY_SUBMISSION_SRC = hits[0].parent
    else:
        # Fallback: the dataset may have kept your files as a .zip
        for z in INPUT_ROOT.glob('datasets/*/*/*.zip'):
            with zipfile.ZipFile(z) as zf:
                if any(n.endswith('agent.yaml') for n in zf.namelist()):
                    extract_dir = Path('/tmp/my_submission_src')
                    shutil.rmtree(extract_dir, ignore_errors=True)
                    zf.extractall(extract_dir)
                    found = list(extract_dir.rglob('agent.yaml'))
                    MY_SUBMISSION_SRC = found[0].parent
                    break

if MY_SUBMISSION_SRC is None:
    listing = sorted(str(p) for p in Path('/kaggle/input').glob('*/*')) + \
              sorted(str(p) for p in Path('/kaggle/input').glob('datasets/*/*'))
    raise FileNotFoundError(
        'Could not find agent.yaml in any attached dataset. '
        'Attach your submission dataset in the Input panel. Found:\n' + '\n'.join(listing)
    )

print(f'Using submission source: {MY_SUBMISSION_SRC}')

# Copy YOUR submission into a writable directory (folder name kept so other cells still work)
AGENT_DIR = WORKING_DIR / 'sample_submission'
if AGENT_DIR.exists():
    shutil.rmtree(AGENT_DIR)
shutil.copytree(MY_SUBMISSION_SRC, AGENT_DIR)

# Keep stray metadata out of the scored zip
(AGENT_DIR / 'dataset-metadata.json').unlink(missing_ok=True)

# Sanity checks
assert (AGENT_DIR / 'agent.yaml').exists(), 'agent.yaml is not at the top level of the submission'
assert (AGENT_DIR / 'configs' / 'sampling.yaml').exists(), 'configs/sampling.yaml is missing'

# NOTE: the old code that overwrote configs/sampling.yaml has been removed on purpose.
# Your repo's configs/sampling.yaml is now the source of truth.

# Load tasks from the published dataset
TASKS_PATH = DATA_DIR / 'tasks.jsonl'
tasks = load_tasks(TASKS_PATH)

print(f'Data directory: {DATA_DIR}')
print(f'Agent directory: {AGENT_DIR}')
print(f'Loaded {len(tasks)} tasks from {TASKS_PATH.name}')
for t in tasks[:5]:
    print(f'  - {t.instance_id} ({t.repo} @ {t.base_commit[:8]})')

Using submission source: /kaggle/input/datasets/reidchilds/gemma-agent-submission
Data directory: /kaggle/input/competitions/gemma-4-developer-agent
Agent directory: /kaggle/working/sample_submission
Loaded 129 tasks from tasks.jsonl
  - fastapi_15661 (fastapi/fastapi @ ee22a4b8)
  - fastapi_15588 (fastapi/fastapi @ cb83b83d)
  - fastapi_15589 (fastapi/fastapi @ 22b02e26)
  - fastapi_15030 (fastapi/fastapi @ 48d58ae3)
  - fastapi_14962 (fastapi/fastapi @ 1e78a36b)


## TESTING. Check for config file

In [3]:
"""
print('source dataset folder:', MY_SUBMISSION_SRC)
print('\n--- eval_config.yaml in the dataset folder ---')
print((Path(MY_SUBMISSION_SRC) / 'eval_config.yaml').read_text())
print('--- eval_config.yaml in AGENT_DIR (what the run used) ---')
print((AGENT_DIR / 'eval_config.yaml').read_text())
print('--- values cell 12 parsed ---')
print(timeout_seconds, max_tool_calls, max_time_minutes, max_turns)
print('--- limits ---')
print(limits)
print(gen_constraints)
"""

"\nprint('source dataset folder:', MY_SUBMISSION_SRC)\nprint('\n--- eval_config.yaml in the dataset folder ---')\nprint((Path(MY_SUBMISSION_SRC) / 'eval_config.yaml').read_text())\nprint('--- eval_config.yaml in AGENT_DIR (what the run used) ---')\nprint((AGENT_DIR / 'eval_config.yaml').read_text())\nprint('--- values cell 12 parsed ---')\nprint(timeout_seconds, max_tool_calls, max_time_minutes, max_turns)\nprint('--- limits ---')\nprint(limits)\nprint(gen_constraints)\n"

## TESTING. Print the file tree

In [4]:
from pathlib import Path

root = Path('/kaggle/working/sample_submission')
assert root.exists(), 'Folder not found. Run the install cell and then the data cell first.'

TEXT_EXT = {'.yaml', '.yml', '.md', '.txt', '.json', '.py', '.toml'}

# File tree with sizes
for p in sorted(root.rglob('*')):
    depth = len(p.relative_to(root).parts) - 1
    size = '' if p.is_dir() else f'  ({p.stat().st_size:,} bytes)'
    print('  ' * depth + ('📁 ' if p.is_dir() else '📄 ') + p.name + size)

# Contents of small text files (config, prompts, skills)
for p in sorted(root.rglob('*')):
    if p.is_file() and p.suffix in TEXT_EXT and p.stat().st_size < 20_000:
        print(f'\n{"=" * 20} {p.relative_to(root)} {"=" * 20}')
        print(p.read_text(errors='replace'))

📁 adapters
  📁 main_lora
    📄 adapter_config.json  (664 bytes)
    📄 adapter_model.safetensors  (217,672 bytes)
  📁 tool_lora
    📄 adapter_config.json  (664 bytes)
    📄 adapter_model.safetensors  (217,672 bytes)
📄 agent.yaml  (438 bytes)
📁 configs
  📄 sampling.yaml  (120 bytes)
📄 eval_config.yaml  (233 bytes)
📁 prompts
  📄 analyzer.md  (527 bytes)
  📄 system.md  (3,668 bytes)
📁 sub_agents
  📄 code_analyzer.yaml  (361 bytes)

==================== adapters/main_lora/adapter_config.json ====================
{
  "alpha_pattern": {},
  "auto_mapping": null,
  "base_model_name_or_path": "google/gemma-4-31b-it-qat-w4a16-ct",
  "bias": "none",
  "fan_in_fan_out": false,
  "inference_mode": true,
  "init_lora_weights": true,
  "layer_replication": null,
  "layers_pattern": null,
  "layers_to_transform": [
    0
  ],
  "loftq_config": {},
  "lora_alpha": 8,
  "lora_dropout": 0.0,
  "megatron_config": null,
  "megatron_core": "megatron.core",
  "modules_to_save": null,
  "peft_type": "LORA",
 

## 3. Code Graph Functions

In [5]:
from swegemma import graph as sg

sample_task = tasks[0]
GRAPH_DIR = str(DATA_DIR / 'graphs')
EMBEDDINGS_DIR = str(DATA_DIR / 'embeddings')

# Load the pre-computed repository code graph and node embeddings for the sample task
repo_graph = sg.get_graph(
    repo_name=sample_task.repo,
    graph_dir=GRAPH_DIR,
    embeddings_dir=EMBEDDINGS_DIR,
    base_commit=sample_task.base_commit,
)
sample_nodes = list(repo_graph.nodes())
print(f'Graph for {sample_task.repo} ({sample_task.instance_id}):')
print(f'  Nodes: {repo_graph.number_of_nodes()}, Edges: {repo_graph.number_of_edges()}')

# Select a connected symbol and query structural neighbors (callers, callees, definitions)
query_node = next((n for n, deg in repo_graph.degree() if deg >= 5), sample_nodes[0])
neighbors = sg.get_neighbor(
    node=query_node,
    graph=repo_graph,
    max_neighbors=10,
)
print(f'\nNeighbors of {query_node!r} (showing up to 5):')
for n in neighbors[:5]:
    print(f'  - {n}')

# Search for semantically similar code nodes using pre-computed vector embeddings
similar = sg.get_similar_nodes(
    node=query_node,
    repo_name=sample_task.repo,
    k=5,
    graph=repo_graph,
    graph_dir=GRAPH_DIR,
    embeddings_dir=EMBEDDINGS_DIR,
    base_commit=sample_task.base_commit,
)
print(f'\nTop similar nodes to {query_node!r}:')
for item in similar:
    print(f"  - {item['node_name']} (similarity={item['similarity']:.4f})")

# Extract an induced subgraph connecting the query symbol and its neighbors
focal_nodes = [query_node, *neighbors[:4]]
subgraph = sg.get_induced_subgraph(repo_graph, focal_nodes)
print(
    f'\nInduced subgraph over {len(focal_nodes)} focal nodes: '
    f'{subgraph.number_of_nodes()} nodes, {subgraph.number_of_edges()} edges'
)

Graph for fastapi/fastapi (fastapi_15661):
  Nodes: 4263, Edges: 2430

Neighbors of 'custom_request_and_route.tutorial002_py310.ValidationErrorLoggingRoute.get_route_handler' (showing up to 5):
  - fastapi/fastapi;custom_request_and_route.tutorial002_py310.ValidationErrorLoggingRoute
  - fastapi/fastapi;fastapi.exceptions.HTTPException
  - fastapi/fastapi;fastapi.exceptions.RequestValidationError

Top similar nodes to 'custom_request_and_route.tutorial002_py310.ValidationErrorLoggingRoute.get_route_handler':
  - custom_request_and_route.tutorial002_an_py310.ValidationErrorLoggingRoute.get_route_handler (similarity=1.0000)
  - fastapi.routing.APIRouter.__init__ (similarity=0.9996)
  - fastapi.applications.FastAPI.setup (similarity=0.9993)
  - custom_request_and_route.tutorial002_an_py310.ValidationErrorLoggingRoute (similarity=0.9991)
  - custom_request_and_route.tutorial002_py310.ValidationErrorLoggingRoute (similarity=0.9991)

Induced subgraph over 4 focal nodes: 4 nodes, 6 edges


## 4. Start vLLM Server

In [6]:
import litellm
import torch
import yaml
from pathlib import Path
from adk_submission import VllmConfig, VllmServer, discover_adapters
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS

litellm.drop_params = True

TARGET_MODEL_NAME = 'gemma-4-31b-it-qat-w4a16-ct'
MODEL_PATH = Path('/kaggle/input/models/google/gemma-4/other/gemma-4-31b-it-qat-w4a16-ct/2')
INFERENCE_API_KEY = 'EMPTY'


# YAML loader that tolerates the competition's custom tags such as "!include"
class TolerantLoader(yaml.SafeLoader):
    pass


def _any_custom_tag(loader, tag_suffix, node):
    if isinstance(node, yaml.ScalarNode):
        return loader.construct_scalar(node)
    if isinstance(node, yaml.SequenceNode):
        return loader.construct_sequence(node)
    return loader.construct_mapping(node)


TolerantLoader.add_multi_constructor('!', _any_custom_tag)

# validate_single_declared_model no longer exists in this swegemma version,
# so check the declared base model directly from the agent and sub-agent yaml files
declared_models = set()
yaml_files = [AGENT_DIR / 'agent.yaml'] + sorted((AGENT_DIR / 'sub_agents').glob('*.yaml'))
for yf in yaml_files:
    cfg = yaml.load(yf.read_text(), Loader=TolerantLoader) or {}
    if isinstance(cfg, dict) and cfg.get('model'):
        declared_models.add(str(cfg['model']))
assert len(declared_models) == 1, f'Expected exactly one declared base model, found: {declared_models}'
declared_model = declared_models.pop()
assert declared_model == TARGET_MODEL_NAME, f'Unexpected model: {declared_model}'
print('Declared base model:', declared_model)

# Discover any PEFT LoRA adapters in the submission directory
adapters = discover_adapters(str(AGENT_DIR), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)

gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 1
tp_size = 4 if gpu_count >= 4 else (2 if gpu_count >= 2 else 1)

vllm_cfg = VllmConfig(
    model=str(MODEL_PATH),
    port=8000,
    host='127.0.0.1',
    tool_call_parser='gemma4',
    reasoning_parser='gemma4',
    max_model_len=32768,
    dtype='bfloat16' if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else 'auto',
    gpu_memory_utilization=0.90,
    enable_auto_tool_choice=True,
    enable_lora=True,
    max_loras=8,
    max_lora_rank=128,
    tensor_parallel_size=tp_size,
    startup_timeout=60 * 50,
)

os.environ['VLLM_ENGINE_READY_TIMEOUT_S'] = '3000'

server_instance = VllmServer(vllm_cfg, adapter_manifest=adapters)
server_instance.start()
print(f'vLLM server started on {server_instance.base_url} (tp={tp_size})')

# Register model and LoRA adapter aliases for Google ADK
models = server_instance.create_model_registry(
    aliases=list(dict.fromkeys([declared_model, TARGET_MODEL_NAME])),
    model_prefix='openai/',
    api_key=INFERENCE_API_KEY,
)

Declared base model: gemma-4-31b-it-qat-w4a16-ct


ServerStartupError: Inference server did not become healthy within 1200s.
--- Server Log Tail (/tmp/inference_server_8000_omtnm4_2.log) ---
utilization=0.9000 is equivalent to --gpu-memory-utilization=0.7153 without CUDA graph memory profiling. To maintain the same effective KV cache size as before, increase --gpu-memory-utilization to 1.0000.
(Worker_TP1 pid=898) INFO 10-22 00:28:22 [gpu_worker.py:452] CUDA graph memory profiling is enabled (VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS=1). This will become the default in v0.19. The current --gpu-memory-utilization=0.9000 is equivalent to --gpu-memory-utilization=0.7153 without CUDA graph memory profiling. To maintain the same effective KV cache size as before, increase --gpu-memory-utilization to 1.0000.
(Worker_TP0 pid=897) INFO 10-22 00:28:22 [gpu_worker.py:436] Available KV cache memory: 8.55 GiB
(Worker_TP0 pid=897) INFO 10-22 00:28:22 [gpu_worker.py:452] CUDA graph memory profiling is enabled (VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS

## 5. Run Phase 1 Inference and Phase 2 Verification

### 5A. settings, task selection, helpers

In [ ]:
import asyncio
import concurrent.futures
import random
import sys
import time
from collections import defaultdict
from pathlib import Path

import pandas as pd
import yaml
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig
from swegemma.config import EvalConfig, build_submission_limits
from swegemma.evaluate import Evaluator

# ============================ SETTINGS (edit here) ============================
NUM_TASKS = 20           # 2 = smoke test, 20 = real comparison
SELECTION = 'first'      # 'first' = first N tasks (old behavior), 'balanced' = spread across repos
SEED = 7                 # same seed -> same tasks every run
BENCHMARK_IDS = []       # optional: paste instance ids here to pin an exact set
VERBOSE = False          # True prints the agent's steps (long output)
SAVE_LOG = False         # True also writes everything printed to run_log.txt
N_TEST = 120             # approximate hidden test set size
TARGET_HOURS = 9.0       # projected total you want to stay under
# ===========================================================================


def run_sync(coro_or_fn, *args, **kwargs):
    """Run an async coroutine synchronously inside a notebook event loop."""
    fn = (lambda: coro_or_fn(*args, **kwargs)) if callable(coro_or_fn) else (lambda: coro_or_fn)
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None
    if loop is not None and loop.is_running():
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
            return pool.submit(lambda: asyncio.run(fn())).result()
    return asyncio.run(fn())


class Tee:
    """Copy everything printed to a file as well as the notebook."""
    def __init__(self, path, original):
        self.f = open(path, 'w')
        self.original = original
    def write(self, s):
        self.f.write(s)
        self.original.write(s)
    def flush(self):
        self.f.flush()
        self.original.flush()
    def close_file(self):
        self.f.close()


# --- Read evaluation settings from the submission's eval_config.yaml ---
eval_config_file = AGENT_DIR / 'eval_config.yaml'
raw_eval_cfg = yaml.safe_load(eval_config_file.read_text(encoding='utf-8'))
eval_section = raw_eval_cfg.get('evaluation', raw_eval_cfg)

timeout_seconds = int(eval_section.get('timeout_seconds', 300))
max_tool_calls = int(eval_section.get('max_tool_calls', 100))
max_time_minutes = float(eval_section.get('max_time_minutes', 60.0))
turns_raw = eval_section.get('max_turns', eval_section.get('max_llm_calls'))
max_turns = int(turns_raw) if turns_raw is not None else None

limits, gen_constraints = build_submission_limits()

print('eval_config.yaml ->', dict(timeout_seconds=timeout_seconds,
                                  max_tool_calls=max_tool_calls,
                                  max_time_minutes=max_time_minutes,
                                  max_turns=max_turns))
print('limits:', limits)
print('generation constraints:', gen_constraints)

# --- Select the benchmark tasks ---
if BENCHMARK_IDS:
    id_set = set(BENCHMARK_IDS)
    SAMPLE_TASKS = [t for t in tasks if t.instance_id in id_set]
    assert len(SAMPLE_TASKS) == len(BENCHMARK_IDS), 'some benchmark ids were not found'
elif SELECTION == 'first':
    SAMPLE_TASKS = tasks[:NUM_TASKS]
else:
    by_repo = defaultdict(list)
    for t in tasks:
        by_repo[t.repo].append(t)
    rng = random.Random(SEED)
    repos = sorted(by_repo)
    for r in repos:
        rng.shuffle(by_repo[r])
    picked = []
    while len(picked) < NUM_TASKS and any(by_repo.values()):
        for r in repos:
            if by_repo[r] and len(picked) < NUM_TASKS:
                picked.append(by_repo[r].pop())
    SAMPLE_TASKS = picked

print(f'{len(SAMPLE_TASKS)} tasks selected:',
      {r: sum(t.repo == r for t in SAMPLE_TASKS) for r in sorted({t.repo for t in SAMPLE_TASKS})})
Path('/kaggle/working/benchmark_ids.txt').write_text(
    '\n'.join(t.instance_id for t in SAMPLE_TASKS))

### 5B. construct evaluator

In [ ]:
# --- Configure the evaluator (Phase 1: agent patch, Phase 2: verification) ---
eval_config = EvalConfig(
    tasks_path=TASKS_PATH,
    snapshots_dir=DATA_DIR / 'snapshots',
    results_dir=WORKING_DIR / 'results',
    submission_dir=AGENT_DIR,
    models=models,
    sandbox='subprocess',
    timeout_seconds=timeout_seconds,
    max_time_minutes=max_time_minutes,
    max_tool_calls=max_tool_calls,
    max_turns=max_turns,
    limits=limits,
    generation_constraints=gen_constraints,
    adapter_manifest=adapters,
    context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
    events_compaction_config=EventsCompactionConfig(
        compaction_interval=15,
        overlap_size=2,
        token_threshold=14336,
        event_retention_size=5,
    ),
    graph_dir=GRAPH_DIR,
    embeddings_dir=EMBEDDINGS_DIR,
    wheels_dir=DATA_DIR / 'wheels',
    verbose=VERBOSE,
)

evaluator = Evaluator(eval_config)
print('Evaluator ready.')

### 5C. Run Tasks

In [ ]:
predictions = []
run_log = []

_orig_stdout = sys.stdout
tee = None
if SAVE_LOG:
    tee = Tee('/kaggle/working/run_log.txt', _orig_stdout)
    sys.stdout = tee

try:
    for idx, task in enumerate(SAMPLE_TASKS, start=1):
        print(f'[{idx}/{len(SAMPLE_TASKS)}] Evaluating {task.instance_id} ({task.repo})...')
        t0 = time.perf_counter()
        result = run_sync(
            evaluator.evaluate_task,
            task=task,
            task_index=idx,
            total_tasks=len(SAMPLE_TASKS),
        )
        wall_s = time.perf_counter() - t0

        predictions.append({'id': task.instance_id, 'prediction': result.agent_patch or ''})
        run_log.append({
            'id': task.instance_id,
            'repo': task.repo,
            'resolved': bool(result.resolved),
            'patch_chars': len(result.agent_patch or ''),
            'tool_calls': result.tool_calls,
            'duration_s': round(result.duration_seconds, 1),
            'wall_s': round(wall_s, 1),
        })
        print(
            f'  -> resolved={result.resolved}, '
            f'exit_code={result.test_exit_code}, '
            f'patch_chars={len(result.agent_patch or "")}, '
            f'tool_calls={result.tool_calls}, '
            f'duration={result.duration_seconds:.1f}s, wall={wall_s:.1f}s'
        )
finally:
    if tee is not None:
        sys.stdout = _orig_stdout
        tee.close_file()

### 5D. Summary / Projection

In [ ]:
submission_df = pd.DataFrame(predictions, columns=['id', 'prediction'])

summary_df = pd.DataFrame(run_log)
cap_s = max_time_minutes * 60
summary_df['hit_tool_cap'] = summary_df['tool_calls'] >= max_tool_calls
summary_df['hit_time_cap'] = summary_df['duration_s'] >= 0.95 * cap_s
summary_df.to_csv('/kaggle/working/run_summary.csv', index=False)

n = len(summary_df)
print(f'Resolved {int(summary_df.resolved.sum())}/{n} | '
      f'non-empty patches {int((summary_df.patch_chars > 0).sum())}/{n} | '
      f'avg tool calls {summary_df.tool_calls.mean():.1f} | '
      f'avg duration {summary_df.duration_s.mean():.0f}s | '
      f'avg wall {summary_df.wall_s.mean():.0f}s')
print(f'Hit tool-call cap: {int(summary_df.hit_tool_cap.sum())} | '
      f'hit time cap: {int(summary_df.hit_time_cap.sum())}')
display(summary_df)
print('By repo:')
display(summary_df.groupby('repo').agg(
    tasks=('id', 'count'),
    patched=('patch_chars', lambda s: int((s > 0).sum())),
    resolved=('resolved', 'sum'),
    avg_wall_s=('wall_s', 'mean'),
))

# 12-hour projection (wall time includes verification, so this leans high)
projected_h = N_TEST * summary_df['wall_s'].mean() / 3600
status = 'OK' if projected_h <= TARGET_HOURS else 'TOO SLOW - trim time before submitting'
print(f'Projected total for {N_TEST} tasks: {projected_h:.1f} h '
      f'(target <= {TARGET_HOURS} h, hard limit 12 h) -> {status}')

### 5E. Extract the dif-file pairs in folders for analysis

In [ ]:
import re
import shutil
import subprocess
import tarfile
import tempfile
import zipfile
from pathlib import Path

# ============================ SETTINGS (edit here) ============================
SNAPSHOTS_DIR = DATA_DIR / 'snapshots'          # where the repo snapshots live
SNAPSHOT_OVERRIDE = None    # optional: a Path to use as the snapshot for every task
OUT_DIR = Path('/kaggle/working/diff_pairs')    # output folder
APPLY_PATCH = True          # also write patched/ copies by applying the diff to the originals
MAKE_ZIP = True             # also write diff_pairs.zip for easy download
# ===========================================================================


def split_patch(patch):
    """Return a list of (original_path, new_path, is_new_file) for each file in a diff."""
    entries = []
    blocks = re.split(r'(?m)^(?=diff --git )', patch)
    for blk in blocks:
        m = re.match(r'diff --git a/(.+?) b/(.+?)\s*$', blk.split('\n', 1)[0])
        if not m:
            continue
        a_path, b_path = m.group(1), m.group(2)
        is_new = bool(re.search(r'(?m)^new file mode', blk)) or bool(re.search(r'(?m)^--- /dev/null', blk))
        entries.append((a_path, b_path, is_new))
    if not entries:  # plain unified diff without "diff --git" headers
        for m in re.finditer(r'(?m)^--- (?:a/)?(\S+)\n\+\+\+ (?:b/)?(\S+)', patch):
            entries.append((m.group(1), m.group(2), m.group(1) == '/dev/null'))
    return entries


def reconstruct_from_hunks(patch, rel_path):
    """Fallback: rebuild the 'before' side of each hunk (context + removed lines) for one file."""
    out, in_file, in_hunk = [], False, False
    for line in patch.split('\n'):
        if line.startswith('diff --git '):
            in_file = line.startswith(f'diff --git a/{rel_path} ')
            in_hunk = False
            continue
        if not in_file:
            continue
        if line.startswith('@@'):
            in_hunk = True
            out.append(line)
            continue
        if in_hunk and (line.startswith(' ') or line.startswith('-')):
            out.append(line[1:])
    return '\n'.join(out) + '\n' if out else ''


def find_snapshot(task):
    """Best-effort search of SNAPSHOTS_DIR for the snapshot belonging to this task."""
    if SNAPSHOT_OVERRIDE is not None:
        return Path(SNAPSHOT_OVERRIDE)
    base = getattr(task, 'base_commit', '') or ''
    repo = getattr(task, 'repo', '') or ''
    keys = [(task.instance_id, 3), (base, 3), (base[:8], 3),
            (repo.replace('/', '__'), 1), (repo.replace('/', '_'), 1), (repo.split('/')[-1], 1)]
    keys = [(k, w) for k, w in keys if k and len(k) >= 4]
    best, best_score = None, 0
    for p in sorted(SNAPSHOTS_DIR.rglob('*')):
        if len(p.relative_to(SNAPSHOTS_DIR).parts) > 3:
            continue
        name = p.name
        score = max([w for k, w in keys if k in name] or [0])
        if score > best_score or (score == best_score and score > 0 and best is not None
                                  and len(p.parts) < len(best.parts)):
            best, best_score = p, score
    return best


def read_originals(location, rel_paths):
    """Read the given files (bytes) from a snapshot that may be a folder, tar, or zip."""
    found = {}
    if location is None or not location.exists():
        return found
    wanted = set(rel_paths)
    if location.is_dir():
        for rel in wanted:
            cands = [location / rel] + [d / rel for d in location.iterdir() if d.is_dir()]
            for c in cands:
                if c.is_file():
                    found[rel] = c.read_bytes()
                    break
    elif tarfile.is_tarfile(location):
        with tarfile.open(location, 'r:*') as tf:
            for member in tf:
                if not member.isfile():
                    continue
                for rel in wanted:
                    if rel not in found and (member.name == rel or member.name.endswith('/' + rel)):
                        found[rel] = tf.extractfile(member).read()
    elif zipfile.is_zipfile(location):
        with zipfile.ZipFile(location) as zf:
            for name in zf.namelist():
                for rel in wanted:
                    if rel not in found and (name == rel or name.endswith('/' + rel)):
                        found[rel] = zf.read(name)
    return found


# --- Build the folders ---
if OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)
OUT_DIR.mkdir(parents=True)

task_by_id = {t.instance_id: t for t in SAMPLE_TASKS}
status_by_id = {}
if 'summary_df' in globals():
    status_by_id = summary_df.set_index('id').to_dict('index')

index_lines = ['# Diff / original-file pairs', '']

for _, row in submission_df.iterrows():
    tid, patch = row['id'], (row['prediction'] or '')
    task = task_by_id.get(tid)
    folder = OUT_DIR / tid.replace('/', '_')
    folder.mkdir(parents=True, exist_ok=True)
    notes = []

    st = status_by_id.get(tid, {})
    header = f"{tid} | resolved={st.get('resolved', '?')} | hit_time_cap={st.get('hit_time_cap', '?')} | tool_calls={st.get('tool_calls', '?')}"

    if not patch.strip():
        (folder / 'NOTES.txt').write_text('Empty patch: the agent produced no diff for this task.\n')
        index_lines.append(f'- {header} | EMPTY PATCH')
        continue

    (folder / 'patch.diff').write_text(patch)
    entries = split_patch(patch)
    rel_paths = [a for a, b, is_new in entries if not is_new]

    snapshot = find_snapshot(task) if task is not None else None
    notes.append(f'snapshot used: {snapshot}')
    originals = read_originals(snapshot, rel_paths)

    for rel in rel_paths:
        if rel in originals:
            dest = folder / 'original' / rel
            dest.parent.mkdir(parents=True, exist_ok=True)
            dest.write_bytes(originals[rel])
        else:
            notes.append(f'ORIGINAL NOT FOUND in snapshot: {rel} (wrote hunk context instead)')
            dest = folder / 'original_from_diff_context' / (rel + '.txt')
            dest.parent.mkdir(parents=True, exist_ok=True)
            dest.write_text(reconstruct_from_hunks(patch, rel))
    for a, b, is_new in entries:
        if is_new:
            notes.append(f'new file created by the patch (no original): {b}')

    # Optional: apply the diff to a scratch copy of the originals to get the patched versions
    if APPLY_PATCH and originals:
        with tempfile.TemporaryDirectory() as tmp:
            tmp = Path(tmp)
            for rel, data in originals.items():
                p = tmp / rel
                p.parent.mkdir(parents=True, exist_ok=True)
                p.write_bytes(data)
            (tmp / 'p.diff').write_text(patch)
            res = subprocess.run(['git', 'apply', '--unsafe-paths', '--whitespace=nowarn', 'p.diff'],
                                 cwd=tmp, capture_output=True, text=True)
            if res.returncode == 0:
                for a, b, is_new in entries:
                    src = tmp / b
                    if src.is_file():
                        dest = folder / 'patched' / b
                        dest.parent.mkdir(parents=True, exist_ok=True)
                        shutil.copy(src, dest)
            else:
                notes.append('could not apply the diff to the originals: ' + res.stderr.strip()[:300])

    (folder / 'NOTES.txt').write_text('\n'.join(notes) + '\n')
    index_lines.append(f'- {header} | files: {[b for a, b, n in entries]}')

(OUT_DIR / 'INDEX.md').write_text('\n'.join(index_lines) + '\n')
print('\n'.join(index_lines))
print('\nWrote', OUT_DIR)

if MAKE_ZIP:
    shutil.make_archive('/kaggle/working/diff_pairs', 'zip', OUT_DIR)
    print('Also wrote /kaggle/working/diff_pairs.zip')

## 6. Package Submission Archive

In [ ]:
import zipfile
from swegemma.config import ALLOWED_SUBMISSION_EXTENSIONS, MAX_SUBMISSION_SIZE_BYTES

# Create submission.zip with agent.yaml at the root of the archive
zip_base = WORKING_DIR / 'submission'
zip_path = Path(shutil.make_archive(str(zip_base), 'zip', root_dir=AGENT_DIR))

# Validate archive contents against competition constraints
with zipfile.ZipFile(zip_path, 'r') as zf:
    infos = zf.infolist()
    total_size = sum(i.file_size for i in infos)
    assert total_size <= MAX_SUBMISSION_SIZE_BYTES, f'Archive exceeds 3 GiB limit: {total_size}'
    for info in infos:
        if not info.is_dir():
            ext = Path(info.filename).suffix.lower()
            assert ext in ALLOWED_SUBMISSION_EXTENSIONS, f'Disallowed file extension: {info.filename}'

print(f'Created {zip_path} ({zip_path.stat().st_size / (1024 * 1024):.2f} MiB)')